# Estimation of Probabilistic Semi-Markovian Graphs

### *DGP (Data Generation Process)*

In [ ]:
import warnings
warnings.filterwarnings("ignore", module="dowhy")  
warnings.filterwarnings("ignore", module="tqdm")
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
#imports + constants
#dgp
import numpy as np
import pandas as pd
#alg1
import networkx as nx
from itertools import product, islice


q_s = [0.2, 0.5, 0.7, 0.9]
p_s = [0.2, 0.5, 0.7, 0.9]
number_of_obs = 4 # # of observed nodes
obs_nodes = [ "X", "W", "V", "Z"]
seed = 40
np.random.seed(seed)
N = 10000

In [ ]:
def create_dataset(map_adj, N, number_of_obs, obs_nodes):
    map_dataset = {}
    for q in q_s:
        for p in p_s:
            nodes = []
            size = len(map_adj[(q,p)]["adj full matrix"][0])
            full_data = np.empty((size, N))
            for i in range(size):
                parents = map_adj[(q,p)]["adj full matrix"][i, :i].astype(bool)# creates a boolean mask
                if not np.any(parents):
                    full_data[i] = np.where(np.random.rand(N) < 0.5,np.random.normal(-2, 0.5, N),np.random.normal(2, 0.5, N)) 
                else:
                    noise = np.where(np.random.rand(N) < 0.5,np.random.normal(-2, 0.5, N),np.random.normal(2, 0.5, N)) 
                    rows_to_consider = full_data[:i][parents]
                    coef_to_consider_lin = map_adj[(q,p)]["coefficients matrix lin"][i, :i][parents]
                    coef_to_consider_sq = map_adj[(q,p)]["coefficients matrix sq"][i, :i][parents]
                    #squaring = rows_to_consider**2
                    multiplying_sq = coef_to_consider_sq[:, None] * (rows_to_consider**2)# [:, None] reshapes into a column vector'
                    multiplying_lin = coef_to_consider_lin[:, None] * (rows_to_consider)
                    summing = np.sum(multiplying_sq, axis=0) + np.sum(multiplying_lin, axis=0)+ noise # fold two rows into 1 
                    summing = (summing - summing.mean())/ (summing.std() + 1e-12)
                    full_data[i] = summing 
            
            full_data = full_data.T
            for i in range(size-number_of_obs):
                nodes.append(f"U{i}")
            nodes.extend(obs_nodes)
                
            full_dataframe = pd.DataFrame(full_data, columns= nodes)
            observed_dataframe = full_dataframe.drop(columns=[col for col in full_dataframe if "U" in col], inplace=False)
            map_dataset[(q,p)] = {"full dataset": full_dataframe, "observed only":observed_dataframe}
    return map_dataset

Binary Model (Self-Defined Weights)

In [ ]:
# same code as for linear just w/o coefficient matrix
map_adj_1 = {}

for q in q_s:
    for p in p_s:
        m = 0 # number of uncertain edges 

        #directed edges
        probs_for_bernoulli = np.tril(np.random.rand(number_of_obs, number_of_obs), k= -1) # just random floats under hauptdiagonale
        adj_directed = (probs_for_bernoulli >0) & (probs_for_bernoulli < q) # condition for existence of directed edge
        adj_directed = adj_directed.astype(int) # converting from T/F to 1/0
        probs = np.tril(np.random.rand(number_of_obs, number_of_obs), k= -1)
        probs = probs*adj_directed
        m+= len(np.argwhere((probs>0) & (probs<1)))
        
        #bidirected edges
        bidirected_edges = {}
        for i in range(number_of_obs):
            for j in range(number_of_obs):
                if i==j: 
                    continue
                    
                bi_prob = np.random.rand()
               
                if bi_prob>0 and bi_prob < p and (i, j) not in bidirected_edges and (j, i) not in bidirected_edges:
                    edge_prob = np.random.rand()
                    bidirected_edges[(i, j)] = edge_prob
                   
        n = len(bidirected_edges)#number of unobserved nodes
        m+= n # fix to be len(np.argwhere((bidirected_edges>0) & (bidirected_edges<1)))
        adj_full = np.zeros((number_of_obs+ n,number_of_obs+ n)) # initializing with appropriate size , matrix of 1/0
        prob_full = np.zeros((number_of_obs+ n,number_of_obs+ n)) # matrix of probs of edges
        #filling the bootom right block then take first bi_edge , take first column of full matrix, take row with index len(bidirected_edges)+i then len(bidirected_edges)+j and fill out 
        adj_full[n:, n:] = adj_directed
        prob_full[n:, n:] = probs
        for idx, (i,j) in enumerate(bidirected_edges.keys()):
            
            adj_full[n+i][idx] = 1 # skip all unobserved nodes (n+) , idx regulates move to the next unobserved
            adj_full[n+j][idx] = 1

            prob_full[n+i][idx]=bidirected_edges[(i,j)]
            prob_full[n+j][idx]=bidirected_edges[(i,j)]
        coef = np.tril(np.random.uniform(-1,1, (number_of_obs+n, number_of_obs+n)), k= -1) # lower triangular matrix (hauptdiagonale not included k<0), matrix of coefficients
        coef_matrix_lin = adj_full*coef
        coef_2 = np.tril(np.random.uniform(-1,1, (number_of_obs+n, number_of_obs+n)), k= -1) # lower triangular matrix (hauptdiagonale not included k<0), matrix of coefficients
        coef_matrix_sq = adj_full*coef_2
            
        map_adj_1[(q,p)] = {"adj full matrix":adj_full, "coefficients matrix lin":coef_matrix_lin,"coefficients matrix sq":coef_matrix_sq, "probability matrix":prob_full, "number of uncertain edges": m, "number of unobserved":n}


map_dataset_1 = create_dataset(map_adj_1, N, number_of_obs, obs_nodes)


## Algorithm 1

### *Splitting Graph into Subgraphs*

In [ ]:
import scipy.stats as stats
from scipy.stats import differential_entropy
from sklearn.ensemble import GradientBoostingRegressor


def weights (targetColumn, parentsColumns ):
    targetColumn = np.asarray(targetColumn).ravel()

    if parentsColumns.size==0:
        noise = targetColumn - targetColumn.mean()
    else:
        parentsColumns = np.asarray(parentsColumns)
        if parentsColumns.ndim ==1:
            parentsColumns = parentsColumns.reshape(-1,1)
        model = GradientBoostingRegressor()
        model.fit( parentsColumns, targetColumn)
        noise = targetColumn - model.predict(parentsColumns)
    return float(differential_entropy(noise))

In [ ]:
#Non-linear model (quadratic)
map_of_subgraphs_1 = {} # (q,p) -> subgraohs -> subgraph + probability of its existence

for q in q_s:
    for p in p_s:
        uncertain_edges = np.argwhere((map_adj_1[(q,p)]["probability matrix"]>0) & (map_adj_1[(q,p)]["probability matrix"]<1))
        sorted_u_e = uncertain_edges[uncertain_edges[:, 1].argsort()] # sorting by columns so that by increasing index k of uncertain edge we move to next column
        m = map_adj_1[(q,p)]["number of uncertain edges"] #
        n = map_adj_1[(q,p)]["number of unobserved"] #
        subgraphs = []
        for combination in product([True, False], repeat=m): # 2^m combinations , m is the length of each combination, each element is either t or f
            subgraoh_prob = map_adj_1[(q,p)]["probability matrix"].copy()
            subgraph_adj = map_adj_1[(q,p)]["adj full matrix"].copy()
            prob_of_subgraph= 1
            k = 0
            for i,v in enumerate(combination):
                if v==False:
                    if i < n: # unobserved column -> delete both entries
                        prob_of_subgraph *= 1-subgraoh_prob[sorted_u_e[k][0]][sorted_u_e[k][1]] # kth entry
                        subgraoh_prob[:, i] = 0
                        subgraph_adj[:,i] = 0
                        
                        k += 2 # deleted two subsequent entries
                    else:
                        prob_of_subgraph *= 1-subgraoh_prob[sorted_u_e[k][0]][sorted_u_e[k][1]]
                        subgraoh_prob[sorted_u_e[k][0]][sorted_u_e[k][1]] = 0
                        subgraph_adj[sorted_u_e[k][0]][sorted_u_e[k][1]] = 0
                        k+=1
                        
                    
                else:
                    prob_of_subgraph *= subgraoh_prob[sorted_u_e[k][0]][sorted_u_e[k][1]]
                    if i < n:
                        k+=2
                    else: 
                        k+=1
                        
            mydataset = map_dataset_1[(q,p)]["observed only"]
            mynodes = list(map_dataset_1[(q, p)]["full dataset"].columns)
            Z = mynodes.index("Z")
            myparents = subgraph_adj[Z].astype(bool)
            parent_names = [mynodes[j] for j in np.where(myparents)[0]]
            observed_parents = [p for p in parent_names if p in mydataset.columns]
            Pa = mydataset[observed_parents].to_numpy()
            H = weights(mydataset["Z"].to_numpy(), Pa)
            weight = 1/ np.exp(H) 
            
            subgraphs.append({"combination":combination,"adj submatrix":subgraph_adj,"prob submatrix":subgraoh_prob, "prob of subgraph": prob_of_subgraph, "weight":weight})
        map_of_subgraphs_1[(q,p)] = {"subgraphs":subgraphs}
    
                    

## Algorithm 3 

### *Estimating Causal Effect of Subgraphs*

In [ ]:
#Binary Model 1
import statsmodels.api as sm
from collections import defaultdict
import networkx as nx
from dowhy import CausalModel
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import econml
from econml.dml import LinearDML

from sklearn.preprocessing import PolynomialFeatures
from econml.iv.dml import DMLIV

map_of_estimates_and_weights_1 = {}
x_s = list(range(-number_of_obs, number_of_obs+1))
for q in q_s:
    for p in p_s:
        dataset = map_dataset_1[(q,p)]["observed only"]
        estimates_and_weights = {"estimate" : [],"distribution":[], "weight": []}
        #mean_z = dataset["Z"].mean()
        #mean_x = dataset["X"].mean()

        for subgraph in map_of_subgraphs_1[(q,p)]["subgraphs"]:
            graph = nx.from_numpy_array(subgraph["adj submatrix"].T,create_using=nx.DiGraph)
            graph = nx.relabel_nodes(graph, dict(enumerate(map_dataset_1[(q,p)]["full dataset"].columns.tolist())))

            model = CausalModel(data=dataset,treatment="X",outcome="Z",graph=graph)# different model for identification 

            try:
                identifiability = model.identify_effect(proceed_when_unidentifiable=False)# equation instead of estimands 
                estimands = identifiability.estimands
                
                x_target = 2.0
                effect = None
                method_used = None
                if estimands.get("backdoor") is not None:
                    adj_vars = identifiability.get_backdoor_variables()
                    features = ["X"] + adj_vars
                
                    outcome_model = GradientBoostingRegressor()
                    outcome_model.fit(dataset[features], dataset["Z"])
                
                    data_do_x = dataset[features].copy()
                    data_do_x["X"] = x_target
                
                    # 1. Capture the full empirical distribution array
                    baseline_pred = outcome_model.predict(dataset[features])
                    residuals = dataset["Z"] - baseline_pred
                    interventional_distribution = outcome_model.predict(data_do_x) + residuals
                    
                    
                    # 2. Calculate the mean for effect tracking
                    effect = interventional_distribution.mean()
                    
                    # 3. Store BOTH for barycenter map!
                    estimates_and_weights["estimate"].append(effect)
                    estimates_and_weights["distribution"].append(interventional_distribution)

                    
                elif estimands.get("frontdoor") is not None:
                    med_vars = list(identifiability.frontdoor_variables)
                    m = med_vars[0]
                    n_obs = len(dataset)
                    rng = np.random.default_rng()
                
                    # 1. Learn E[M | X]
                    m_model = GradientBoostingRegressor()
                    m_model.fit(dataset[["X"]], dataset[m])
                
                    # Abduction step for M: recover each unit's idiosyncratic residual
                    m_fitted = m_model.predict(dataset[["X"]])
                    m_residuals = dataset[m].values - m_fitted
                
                    # 2. Learn E[Z | M, X]
                    z_model = GradientBoostingRegressor()
                    z_model.fit(dataset[[m, "X"]], dataset["Z"])
                
                    # Abduction step for Z: recover each unit's idiosyncratic residual
                    z_fitted = z_model.predict(dataset[[m, "X"]])
                    z_residuals = dataset["Z"].values - z_fitted
                
                    # 3. Sample M ~ p(M | do(X = x_target))
                    #    point prediction at x_target + a bootstrap draw from M's residual distribution
                    m_point_pred = m_model.predict(pd.DataFrame({"X": [x_target]}))[0]
                    m_samples = m_point_pred + rng.choice(m_residuals, size=n_obs, replace=True)
                
                    # 4. Outer integral over the natural marginal P(X):
                    #    the dataset's own X column IS an empirical sample from P(X)
                    x_prime_samples = dataset["X"].values
                
                    simulated_data = pd.DataFrame({m: m_samples, "X": x_prime_samples})
                    z_point_preds = z_model.predict(simulated_data)
                
                    # 5. Add back a resampled Z-residual so each entry is a sample from
                    #    p(Z | do(X = x_target)), not a conditional mean
                    z_samples = z_point_preds + rng.choice(z_residuals, size=n_obs, replace=True)
                
                    expected_value_do_x = z_samples.mean()
                    estimates_and_weights["estimate"].append(expected_value_do_x)
                    estimates_and_weights["distribution"].append(z_samples)

                    
                elif estimands.get("iv") is not None:
                    iv_vars = identifiability.get_instrumental_variables()
                    iv = iv_vars[0] 
                
                    est_model = DMLIV(
                        model_y_xw=GradientBoostingRegressor(), 
                        model_t_xw=GradientBoostingRegressor(),
                        model_t_xwz=GradientBoostingRegressor(),
                        treatment_featurizer=PolynomialFeatures(degree=2, include_bias=False), 
                    )
                    # Fit the IV model using the observed data
                    est_model.fit(Y=dataset["Z"], T=dataset["X"], Z=dataset[[iv]], X=None)
                
                    # 1. Calculate the individualized causal shift for every sample
                    # T0 is what they actually received; T1 is the intervention target
                    causal_shifts = est_model.effect(X=None, T0=dataset["X"], T1=x_target)
                    
                    # 2. Recover the absolute counterfactual outcomes (The Interventional Distribution)
                    interventional_distribution = dataset["Z"] + causal_shifts
                    
                    # 3. Calculate the expected value (mean)
                    effect = interventional_distribution.mean()
                    
                    # 4. Store BOTH for barycenter map!
                    estimates_and_weights["estimate"].append(effect)
                    estimates_and_weights["distribution"].append(interventional_distribution)
                else:
                    continue
                estimates_and_weights["weight"].append(subgraph["weight"])

            except Exception as e:
                print(f"Skipping subgraph q={q}, p={p}: {e}")
                continue

        map_of_estimates_and_weights_1[(q,p)] = estimates_and_weights



### *Finding Wasserstein Barycenters of Estimates*

In [ ]:
import ot # another library for Wasserstein Barycenters
import numpy as np

def make_grid(sample_arrays, nbins=100, lo_pct=0.5, hi_pct=99.5, pad_frac=0.05):
    pooled = np.concatenate([np.asarray(a).ravel() for a in sample_arrays])
    pooled = pooled[np.isfinite(pooled)]
    lo, hi = np.percentile(pooled, [lo_pct, hi_pct])      # robust to the heavy tails
    if hi <= lo: hi = lo + 1.0
    pad = pad_frac * (hi - lo)
    return np.linspace(lo - pad, hi + pad, nbins), lo - pad, hi + pad

def to_hist(samples, lo, hi, nbins=100):
    h, _ = np.histogram(np.asarray(samples).ravel(), bins=nbins, range=(lo, hi))
    h = h.astype(float); s = h.sum()
    return h / s if s > 0 else h

def barycenters(map_of_estimates_and_weights):
    
    regularization = 1e-3

    n = 100 # number of bins
    
    
    
    
    map_of_barycenters = {}
    
    for q in q_s:
        for p in p_s:
            
            distributions = []#among all subgraphs of a particular pair of q and p
            if map_of_estimates_and_weights[(q,p)]["estimate"] == []:
                map_of_barycenters[(q,p)] =[]
                continue
            for distribution in map_of_estimates_and_weights[(q,p)]["distribution"]: # e is an estimate with corresponding weight#
                
                distributions.append(distribution)
                
            samples = list(distributions)
             
            y, low, high = make_grid(samples, 100)
            M = ot.dist(y.reshape(-1, 1), y.reshape(-1, 1), metric='sqeuclidean')# sqeuclidean -> squared wasserstein2 distance  
            M /= M.max()
            A = np.stack([to_hist(d, low, high, 100) for d in distributions]).T
            weights = np.array([e for e in map_of_estimates_and_weights[(q,p)]["weight"]])
            weights /= weights.sum()
            
            
            barycenter = ot.bregman.barycenter(A,M, regularization, weights)# check only for gaussian
            barycenter /= barycenter.sum()
            map_of_barycenters[(q,p)] = {"barycenter":barycenter, "A": A, "grid": y, "lo": low, "hi": high}
    return map_of_barycenters
    


 
map_of_barycenters_1 = barycenters(map_of_estimates_and_weights_1)
        


In [ ]:
map_of_target = {}

for q in q_s:
    for p in p_s:
        dataset = map_dataset_1[(q, p)]["full dataset"]
        target = {"estimate": None, "distribution": None}

        adj_full_matrix = map_adj_1[(q, p)]["adj full matrix"]
        graph = nx.from_numpy_array(adj_full_matrix.T, create_using=nx.DiGraph)
        graph = nx.relabel_nodes(graph, dict(enumerate(map_dataset_1[(q, p)]["full dataset"].columns.tolist())))

        model = CausalModel(data=dataset, treatment="X", outcome="Z", graph=graph)

        try:
            identifiability = model.identify_effect(proceed_when_unidentifiable=False)

            if identifiability.no_directed_path:
                
                target["estimate"] = 0.0
                target["distribution"] = np.zeros(len(dataset))

            elif identifiability.estimands is None:
                print(f"No identification result for full graph q={q}, p={p}")

            else:
                estimands = identifiability.estimands
                x_target = 2.0

                if estimands.get("backdoor") is not None:
                    adj_vars = identifiability.get_backdoor_variables()  # was: list(identifiability.backdoor_variables)
                    features = ["X"] + adj_vars

                    outcome_model = GradientBoostingRegressor()
                    outcome_model.fit(dataset[features], dataset["Z"])

                    data_do_x = dataset[features].copy()
                    data_do_x["X"] = x_target

                    baseline_pred = outcome_model.predict(dataset[features])
                    residuals = dataset["Z"] - baseline_pred
                    interventional_distribution = outcome_model.predict(data_do_x) + residuals
                    effect = interventional_distribution.mean()

                    target["estimate"] = effect
                    target["distribution"] = interventional_distribution

                elif estimands.get("frontdoor") is not None:
                    med_vars = list(identifiability.frontdoor_variables)
                    m = med_vars[0]
                    n_obs = len(dataset)
                    rng = np.random.default_rng()
                
                    # 1. Learn E[M | X]
                    m_model = GradientBoostingRegressor()
                    m_model.fit(dataset[["X"]], dataset[m])
                
                    # Abduction step for M: recover each unit's idiosyncratic residual
                    m_fitted = m_model.predict(dataset[["X"]])
                    m_residuals = dataset[m].values - m_fitted
                
                    # 2. Learn E[Z | M, X]
                    z_model = GradientBoostingRegressor()
                    z_model.fit(dataset[[m, "X"]], dataset["Z"])
                
                    # Abduction step for Z: recover each unit's idiosyncratic residual
                    z_fitted = z_model.predict(dataset[[m, "X"]])
                    z_residuals = dataset["Z"].values - z_fitted
                
                    # 3. Sample M ~ p(M | do(X = x_target))
                    #    point prediction at x_target + a bootstrap draw from M's residual distribution
                    m_point_pred = m_model.predict(pd.DataFrame({"X": [x_target]}))[0]
                    m_samples = m_point_pred + rng.choice(m_residuals, size=n_obs, replace=True)
                
                    # 4. Outer integral over the natural marginal P(X):
                    #    the dataset's own X column IS an empirical sample from P(X)
                    x_prime_samples = dataset["X"].values
                
                    simulated_data = pd.DataFrame({m: m_samples, "X": x_prime_samples})
                    z_point_preds = z_model.predict(simulated_data)
                
                    # 5. Add back a resampled Z-residual so each entry is a sample from
                    #    p(Z | do(X = x_target)), not a conditional mean
                    z_samples = z_point_preds + rng.choice(z_residuals, size=n_obs, replace=True)
                
                    expected_value_do_x = z_samples.mean()
                    target["estimate"]= expected_value_do_x
                    target["distribution"]= z_samples

                elif estimands.get("iv") is not None:
                    iv_vars = identifiability.get_instrumental_variables()  # was: list(identifiability.instrumental_variables)
                    iv = iv_vars[0]

                    est_model = DMLIV(
                        model_y_xw=GradientBoostingRegressor(),
                        model_t_xw=GradientBoostingRegressor(),
                        model_t_xwz=GradientBoostingRegressor(),
                        treatment_featurizer=PolynomialFeatures(degree=2, include_bias=False),
                    )
                    est_model.fit(Y=dataset["Z"], T=dataset["X"], Z=dataset[[iv]], X=None)

                    causal_shifts = est_model.effect(X=None, T0=dataset["X"], T1=x_target)
                    interventional_distribution = dataset["Z"] + causal_shifts
                    effect = interventional_distribution.mean()

                    target["estimate"] = effect
                    target["distribution"] = interventional_distribution

                else:
                    print(f"No identified estimand for full graph q={q}, p={p}")

        except Exception as e:
            print(f"Skipping full graph q={q}, p={p}: {e}")

        map_of_target[(q, p)] = target

In [ ]:
import numpy as np
import ot

def to_hist(samples, lo, hi, nbins):
    h, _ = np.histogram(np.asarray(samples).ravel(), bins=nbins, range=(lo, hi))
    h = h.astype(float)
    s = h.sum()
    return h / s if s > 0 else h

map_of_errors = {}
map_of_target_distr = {}

for q in q_s:
    for p in p_s:
        bary_data   = map_of_barycenters_1.get((q, p))
        target_data = map_of_target.get((q, p))

        # skip unidentifiable / missing on either side
        if (not bary_data) or bary_data == [] \
           or target_data is None or target_data.get("distribution") is None:
            continue

        # the SAME grid the barycenter (and its subgraph histograms) were built on
        y, lo, hi = bary_data["grid"], bary_data["lo"], bary_data["hi"]
        nbins = len(y)

        M = ot.dist(y.reshape(-1, 1), y.reshape(-1, 1), metric="sqeuclidean")
        M /= M.max()

        # histogram the target's actual interventional samples on that grid
        target_hist = to_hist(target_data["distribution"], lo, hi, nbins)
        if target_hist.sum() == 0:          # target fell entirely outside the grid
            continue

        w2 = np.sqrt(ot.emd2(target_hist, bary_data["barycenter"], M))
        map_of_errors[(q, p)]       = w2
        map_of_target_distr[(q, p)] = target_hist



In [ ]:

import matplotlib.pyplot as plt
import numpy as np

# 1. configurations
qp_pairs = [(q, p) for q in q_s for p in p_s]

# Dynamically size the grid (e.g., 4x4 if there are 16 pairs)
nrows = len(q_s)
ncols = len(p_s)

fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=(ncols * 5, nrows * 4), 
    tight_layout=True,
    squeeze=False # axes is always a predictable 2D matrix
)

# Flatten the axes matrix for easy 1D iteration
axes = axes.flatten()

# 2. Iterate through every combination
for j, (q, p) in enumerate(qp_pairs):
    ax = axes[j]
    
    # Safely retrieve data from both dictionaries
    bary_data = map_of_barycenters_1.get((q, p))
    target_data = map_of_target.get((q, p))
    
    #  Check if either the target or barycenter failed
    if not bary_data or not target_data or target_data["distribution"] is None or bary_data == []:
        ax.axis('off')
        ax.text(0.5, 0.5, f"Unidentifiable\nq={q}, p={p}", 
                ha='center', va='center', fontsize=14, color='red')
        continue
        
    barycenter = bary_data["barycenter"]
    A_matrix = bary_data["A"] # Matrix where each column is a subgraph PMF
    
    
    y = bary_data["grid"]
    lo = bary_data["lo"]
    hi = bary_data["hi"]
    n_bins = len(y)
    bar_width = (hi - lo) / n_bins
    
    # 3. Process the True Target Distribution
    raw_distribution = target_data["distribution"]
    
    if len(raw_distribution) > 0 and isinstance(raw_distribution[0], (list, np.ndarray)):
        raw_distribution = raw_distribution[0]
        
    raw_distribution = np.array(raw_distribution)
    raw_distribution = raw_distribution[~np.isnan(raw_distribution)] 
    
   
    target_hist, _ = np.histogram(raw_distribution, bins=n_bins, range=(lo, hi))
    target_hist = target_hist.astype(float)
    
    if target_hist.sum() > 0:
        target_hist /= target_hist.sum()
    else:
        ax.axis('off')
        ax.text(0.5, 0.5, f"Target Out of Bounds\nq={q}, p={p}", 
                ha='center', va='center', fontsize=14, color='orange')
        continue

    # 4. Plot Layer 1: Subgraph Distributions
    num_subgraphs = A_matrix.shape[1]
    for i in range(num_subgraphs):
        
        label = "Subgraphs" if i == 0 else None
        ax.plot(y, A_matrix[:, i], color='gray', alpha=0.2, label=label)
        
    # 5. Plot Layer 2: True Target Distribution
    ax.bar(y, target_hist, width=bar_width, color="blue", alpha=0.4, label="True Target")
    
    # 6. Plot Layer 3: Barycenter
    ax.plot(y, barycenter, color='orange', linewidth=2.5, label="Barycenter")
    
    # 7. Metadata and Dynamic Scaling
    max_peak = max(target_hist.max(), barycenter.max(), A_matrix.max())
    
    ax.set_title(f"Distribution q={q}, p={p} | do(X=2.0). Error: {map_of_errors[(q,p)]:.4f}")
    
    
    ax.set_xlim(lo, hi)
    ax.set_ylim(0, min(1.05, max_peak * 1.1)) 
    
    ax.set_ylabel("Probability Mass")
    ax.set_xlabel("Estimated Outcome")
    ax.legend(loc="upper right")

plt.show()